### 2.3.4 具体的分割器
LangChain提供了许多不同类型的文档切分器：https://docs.langchain.com/oss/python/integrations/splitters

#### 2.3.4.1 CharacterTextSplitter：Split by character


类签名:

```python
CharacterTextSplitter(
  self,
  separator: str = '\n\n',
  is_separator_regex: bool = False,
  kwargs: Any = {}
)
```

* chunk_size ：每个切块的最大字符数量，默认值为4000。
*  chunk_overlap ：相邻两个切块之间的最大重叠字符数量，默认值为200。为了保证段之间语义完整，可以设置每个块之间有一部分重叠。
* separator ：分割使用的分隔符，默认值为"\n\n"。
* length_function ：用于计算切块长度的方法。默认赋值为父类TextSplitter的len函数。

In [2]:
# 1.导入相关依赖
from langchain_text_splitters import CharacterTextSplitter

# 2.示例文本
text = """
LangChain 是一个用于开发由语言模型驱动的应用程序的框架的。它提供了一套工具和抽象，使开发者能够更容易地构建复杂的应用程序。简单来说，原生的 LLM（如 GPT、Claude、DeepSeek 等）只负责根据输入的文本生成回答。但在实际开发中，AI 需要连接外部世界——比如查阅企业的私有数据库、搜索互联网、调用 API、或者记住多轮对话的上下文。LangChain 就充当了“大模型的操作系统或胶水”，把模型与外部数据、工具和工作流串联起来。
"""

# 3.定义字符分割器
splitter = CharacterTextSplitter(
    chunk_size=50, # 每块大小
    chunk_overlap=5,# 块与块之间的重复字符数
    # length_function=len,
    separator=""   # 设置为空字符串时，表示禁用分隔符优先
)

# 4.分割文本
texts = splitter.split_text(text)

# 5.打印结果
for i, chunk in enumerate(texts):
    print(f"块 {i+1}:长度：{len(chunk)}")
    print(repr(chunk))
    print("-" * 50)

块 1:长度：49
'LangChain 是一个用于开发由语言模型驱动的应用程序的框架的。它提供了一套工具和抽象，使开发'
--------------------------------------------------
块 2:长度：50
'象，使开发者能够更容易地构建复杂的应用程序。简单来说，原生的 LLM（如 GPT、Claude、De'
--------------------------------------------------
块 3:长度：50
'de、DeepSeek 等）只负责根据输入的文本生成回答。但在实际开发中，AI 需要连接外部世界——'
--------------------------------------------------
块 4:长度：50
'部世界——比如查阅企业的私有数据库、搜索互联网、调用 API、或者记住多轮对话的上下文。LangCh'
--------------------------------------------------
块 5:长度：47
'angChain 就充当了“大模型的操作系统或胶水”，把模型与外部数据、工具和工作流串联起来。'
--------------------------------------------------


第一个块不足50个字符是因为，原本 `text` 开头存在换行符。而`CharacterTextSplitter`存在默认参数`strip_whitespace=True`会对每个 chunk 调用 .strip()，删除两端空白。删掉换行后剩下 49 个字符，不会再补一个字符进来。

In [3]:
# 1.导入相关依赖
from langchain_text_splitters import CharacterTextSplitter

# 2.示例文本
text = """
LangChain 是一个用于开发由语言模型驱动的应用程序的框架的。它提供了一套工具和抽象，使开发者能够更容易地构建复杂的应用程序。简单来说，原生的 LLM（如 GPT、Claude、DeepSeek 等）只负责根据输入的文本生成回答。但在实际开发中，AI 需要连接外部世界——比如查阅企业的私有数据库、搜索互联网、调用 API、或者记住多轮对话的上下文。LangChain 就充当了“大模型的操作系统或胶水”，把模型与外部数据、工具和工作流串联起来。
"""

# 3.定义字符分割器
splitter = CharacterTextSplitter(
    chunk_size=50, # 每块大小
    chunk_overlap=5,# 块与块之间的重复字符数
    # length_function=len,
    separator="。"   # 设置为空字符串时，表示禁用分隔符优先
)

# 4.分割文本
texts = splitter.split_text(text)

# 5.打印结果
for i, chunk in enumerate(texts):
    print(f"块 {i+1}:长度：{len(chunk)}")
    print(repr(chunk))
    print("-" * 50)

Created a chunk of size 51, which is longer than the specified 50
Created a chunk of size 59, which is longer than the specified 50


块 1:长度：33
'LangChain 是一个用于开发由语言模型驱动的应用程序的框架的'
--------------------------------------------------
块 2:长度：31
'它提供了一套工具和抽象，使开发者能够更容易地构建复杂的应用程序'
--------------------------------------------------
块 3:长度：51
'简单来说，原生的 LLM（如 GPT、Claude、DeepSeek 等）只负责根据输入的文本生成回答'
--------------------------------------------------
块 4:长度：59
'但在实际开发中，AI 需要连接外部世界——比如查阅企业的私有数据库、搜索互联网、调用 API、或者记住多轮对话的上下文'
--------------------------------------------------
块 5:长度：48
'LangChain 就充当了“大模型的操作系统或胶水”，把模型与外部数据、工具和工作流串联起来。'
--------------------------------------------------


separator优先原则：当设置了 `separator` （如"。"），分割器会首先尝试在分隔符处分割，然后再考虑 chunk_size。这是为了避免在句子中间硬性切断。这种设计是为了：
1. 优先保持语义完整性（不切断句子）
2. 避免产生无意义的碎片（如半个单词/不完整句子）
3. 如果 chunk_size 比片段小，无法拆分片段，导致 overlap失效。也就是说，片段足够长才能overlap
4. chunk_overlap仅在合并后的片段之间生效（如果 chunk_size 足够大）。如果没有合并的片段，则 overlap失效。

说明：若必须禁用分隔符（如处理无空格文本），需容忍实际块长略小于 chunk_size （尤其对中文）

In [4]:
# 1.导入相关依赖
from langchain_text_splitters import CharacterTextSplitter

# 2.定义要分割的文本
text = "这是第一段文本。这是第二段内容。最后一段结束。"

# 3.定义字符分割器
text_splitter = CharacterTextSplitter(
    chunk_size=20,
    chunk_overlap=8,
    separator="。",
    # keep_separator=True #chunk中是否保留切割符
)

# 4.分割文本
chunks = text_splitter.split_text(text)

# 5.打印结果
for  i,chunk in enumerate(chunks):
    print(f"块 {i + 1}:长度：{len(chunk)}")
    print(chunk)
    print("-"*50)

块 1:长度：15
这是第一段文本。这是第二段内容
--------------------------------------------------
块 2:长度：14
这是第二段内容。最后一段结束
--------------------------------------------------
